# Modelo 6 — Árvore de Decisão para Predição de Falhas em Portas de Plataforma

Este notebook tem como objetivo desenvolver um modelo de árvore de decisão capaz de estimar se uma Porta de Plataforma apresentará uma falha corretiva emergencial nos próximos 7 dias.

Para isso, serão utilizados:

- O histórico limpo de alarmes das portas como fonte das variáveis de entrada;
- As ordens corretivas emergenciais ZM01 como indicação da ocorrência de falhas;
- Informações temporais, operacionais e históricas de cada porta.

O desenvolvimento será dividido nas seguintes etapas:

1. Importação das bibliotecas;
2. Carregamento dos dados;
3. Análise e preparação das bases;
4. Identificação única das portas;
5. Construção das variáveis preditoras;
6. Construção do target;
7. Separação temporal dos dados;
8. Treinamento da árvore de decisão;
9. Avaliação do modelo;
10. Interpretação dos resultados.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Localiza a raiz do projeto, independentemente de onde o Jupyter foi iniciado
diretorio_atual = Path.cwd().resolve()

if (diretorio_atual / "data").exists():
    raiz_projeto = diretorio_atual
elif (diretorio_atual.parent / "data").exists():
    raiz_projeto = diretorio_atual.parent
elif (diretorio_atual.parent.parent / "data").exists():
    raiz_projeto = diretorio_atual.parent.parent
else:
    raise FileNotFoundError("Não foi possível localizar a pasta data do projeto.")

caminho_alarmes = (
    raiz_projeto
    / "data"
    / "Alarmes"
    / "alarmes_trata_limpos_consolidado.csv"
)

caminho_zm01 = (
    raiz_projeto
    / "data"
    / "Corretiva Emergencial"
    / "processados"
    / "ZM01_limpa.xlsx"
)

print("Raiz do projeto:", raiz_projeto)
print("Base de alarmes encontrada:", caminho_alarmes.exists())
print("Base ZM01 encontrada:", caminho_zm01.exists())

Raiz do projeto: /home/kaian/Documentos/modulo7/g04
Base de alarmes encontrada: True
Base ZM01 encontrada: True


In [7]:
colunas_alarmes = [
    "TAG",
    "dt_alarme",
    "duracao_calculada_seg",
    "estacao_extraida",
    "plataforma_extraida",
    "porta_extraida",
    "Alarme"
]

alarmes = pd.read_csv(
    caminho_alarmes,
    usecols=colunas_alarmes,
    parse_dates=["dt_alarme"],
    low_memory=False
)

# Padroniza o nome da coluna para facilitar as próximas etapas
alarmes = alarmes.rename(
    columns={
        "Alarme": "tipo_alarme"
    }
)

print(f"Quantidade de registros: {len(alarmes):,}")
print(f"Quantidade de colunas: {alarmes.shape[1]}")

display(alarmes.head())

Quantidade de registros: 742,506
Quantidade de colunas: 7


,TAG,tipo_alarme,dt_alarme,duracao_calculada_seg,estacao_extraida,plataforma_extraida,porta_extraida
0,PSD_FAL_FAL2_PDM09_OBS,OBS,2026-05-12 08:16:52.657,1.013,FAL,FAL2,9.0
1,PSD_LUZ_LUZ1_PDM08_OBS,OBS,2026-05-12 08:15:09.913,1.004,LUZ,LUZ1,8.0
2,PSD_FAL_FAL1_PDM15_OBS,OBS,2026-05-12 08:14:06.657,1.013,FAL,FAL1,15.0
3,PSD_LUZ_LUZ1_PDM19_OBS,OBS,2026-05-12 08:09:31.930,0.987,LUZ,LUZ1,19.0
4,PSD_FRE_FRE1_PDM14_OBS,OBS,2026-05-12 08:05:05.830,9.017,FRE,FRE1,14.0


In [8]:
print("Dimensão da base:")
print(f"Linhas: {alarmes.shape[0]:,}")
print(f"Colunas: {alarmes.shape[1]}")

print("\nPeríodo dos alarmes:")
print("Data inicial:", alarmes["dt_alarme"].min())
print("Data final:", alarmes["dt_alarme"].max())

print("\nTipos das colunas:")
display(alarmes.dtypes.to_frame("tipo"))

print("\nValores ausentes:")
ausencias_alarmes = (
    alarmes.isna()
    .sum()
    .to_frame("quantidade_ausente")
)

ausencias_alarmes["percentual_ausente"] = (
    ausencias_alarmes["quantidade_ausente"]
    / len(alarmes)
    * 100
).round(2)

display(ausencias_alarmes)

Dimensão da base:
Linhas: 742,506
Colunas: 7

Período dos alarmes:
Data inicial: 2023-02-27 06:26:56.897000
Data final: 2026-05-12 08:16:52.657000

Tipos das colunas:


,tipo
TAG,str
tipo_alarme,str
dt_alarme,datetime64[us]
duracao_calculada_seg,float64
estacao_extraida,str
plataforma_extraida,str
porta_extraida,float64



Valores ausentes:


,quantidade_ausente,percentual_ausente
TAG,0,0.00
tipo_alarme,0,0.00
dt_alarme,0,0.00
duracao_calculada_seg,0,0.00
estacao_extraida,0,0.00
plataforma_extraida,0,0.00
porta_extraida,170,0.02
